# 02 - Exploratory Data Analysis (silver)

Quick checks on the silver OBT after cleaning.
Goal is to verify the cleaning worked and spot any new issues
that only become visible once the data is in shape.

*Some parts and solutions of this code was found through help from Claude*

In [0]:
%reload_ext autoreload
%autoreload 2

import sys
import os

# Make repo root importable so we can use utils/
sys.path.append(os.path.abspath(".."))

from pyspark.sql import functions as F
from utils.constants import SILVER_RESULTS_OBT
from utils.io_helpers import read_table

## Basic shape

Confirm rowcount and schema match what we wrote from silver.

In [0]:
silver_df = read_table(spark, SILVER_RESULTS_OBT)

print(f"Rows in silver: {silver_df.count():,}")
print(f"Columns: {len(silver_df.columns)}")
silver_df.printSchema()

## Event type distribution

The new `event_type` column classifies events into three buckets.
Multi-day events were kept based on instructor feedback rather than dropped.

In [0]:
silver_df.groupBy("event_type").count().orderBy(F.col("count").desc()).show()

## Null counts after cleaning

How does it look compared to bronze? Cleaning should have removed the worst offenders.

In [0]:
null_counts = silver_df.select([
    F.count(F.when(F.col(c).isNull(), c)).alias(c)
    for c in silver_df.columns
])
null_counts.show(vertical=True, truncate=False)

## Sample rows

Final sanity check that the table looks like we expect.

In [0]:
silver_df.show(5, vertical=True, truncate=False)

## Unique country codes (for Bonus 1)

We need the full list of country codes in silver to build a mapping
to full country names for Bonus 1 (country data enrichment).

In [0]:
unique_countries = silver_df.select("athlete_country").distinct().orderBy("athlete_country")
print(f"Unique countries: {unique_countries.count()}")
unique_countries.show(unique_countries.count(), truncate=False)

## Event date formats (for dim_date design - Bonus 5)

Looking at the parsed start and end dates after building dim_date parsing.
Confirms that the parser handled the different formats in event_dates correctly.

In [0]:
silver_df.select("start_date", "end_date").distinct().limit(30).show(30, truncate=False)

In [0]:
date_range = silver_df.agg(
    F.min("start_date").alias("min_date"),
    F.max("end_date").alias("max_date")
)
date_range.show()